# PII 마스킹 이식

Ch16의 PII 마스킹을 PIIMiddleware로 옮기면 적용 지점과 코드가 어떻게 달라질까요?


## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/02-PII-마스킹-이식.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.


## 패키지 버전 확인


In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비

이 모델은 정해 둔 응답으로 훅 경계를 관찰하는 결정적 검사 장치입니다. `BaseChatModel` 상속 구현 자체는 이 절의 학습 대상이 아닙니다.


In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## Ch16 원본 마스킹 함수 재현

개인 식별 정보(PII)는 이메일, 전화번호처럼 개인을 알아볼 수 있게 하는 데이터입니다. `mask_pii(text)`는 문자열 하나를 받아 정규식과 일치한 부분을 표식으로 바꾸고, 유형별 발견 개수를 함께 반환합니다. 이 함수만 정의해 두면 아무 요청에도 자동 적용되지 않습니다. 호출 지점을 빠뜨린 Before가 그대로 노출되는 이유입니다.

`PIIMiddleware`는 같은 검사를 에이전트 경계에 등록합니다. 입력 검사는 모델에 전달하기 전에 사용자 메시지를 바꾸고, 출력 검사는 모델 응답을 호출자에게 반환하기 전에 바꿉니다. `strategy="mask"`는 일부를 가리는 처리 방식이며, 감지기가 찾지 못한 형식이나 이미지 속 정보까지 보호한다는 뜻은 아닙니다. 입력과 출력 두 방향은 서로 대신할 수 없습니다.


In [3]:
# 이 셀은 Ch16 원본 마스킹 함수 재현 작업을 수행하고 확인할 값을 출력합니다.
import re
PII_PATTERNS = {
 "email": re.compile(r"(?<![\w.])[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}(?![\w.])"),
 "phone": re.compile(r"(?<!\d)01[016789][ -]?\d{3,4}[ -]?\d{4}(?!\d)"),
 "rrn": re.compile(r"(?<!\d)\d{6}[ -]?[1-4]\d{6}(?!\d)"),
 "card": re.compile(r"(?<!\d)(?:\d[ -]?){15}\d(?!\d)"),
}
def mask_pii(text: str) -> tuple[str, dict[str, int]]:
    masked, counts = text, {}
    for name, pattern in PII_PATTERNS.items():
        masked, count = pattern.subn(f"<{name.upper()}_MASKED>", masked)
        counts[name] = count
    return masked, counts
sample = "learner@example.com 010-1234-5678 900101-1234567 1234-5678-9012-3456"
direct_masked, direct_counts = mask_pii(sample)
print(direct_masked); print(direct_counts)
assert direct_counts == {"email":1,"phone":1,"rrn":1,"card":1}

<EMAIL_MASKED> <PHONE_MASKED> <RRN_MASKED> <CARD_MASKED>
{'email': 1, 'phone': 1, 'rrn': 1, 'card': 1}


## Before: 호출을 빠뜨린 경로


In [4]:
# 이 셀은 Before: 호출을 빠뜨린 경로 작업을 수행하고 확인할 값을 출력합니다.
before_model = ScriptedModel(replies=[AIMessage(content="확인: learner@example.com")])
before_agent = create_agent(before_model, tools=[])
before = before_agent.invoke({"messages":[{"role":"user","content":sample}]})
print([message.text for message in before["messages"]])
assert "learner@example.com" in before["messages"][0].text

['learner@example.com 010-1234-5678 900101-1234567 1234-5678-9012-3456', '확인: learner@example.com']


## After: 입력과 출력 미들웨어 적용


In [5]:
# 이 셀은 After: 입력과 출력 미들웨어 적용 작업을 수행하고 확인할 값을 출력합니다.
from langchain.agents.middleware import PIIMiddleware
after_model = ScriptedModel(replies=[AIMessage(content="확인: learner@example.com")])
after_agent = create_agent(after_model, tools=[], middleware=[
    PIIMiddleware("email", strategy="mask", apply_to_input=True, apply_to_output=True)
])
after = after_agent.invoke({"messages":[{"role":"user","content":"회신: learner@example.com 입니다."}]})
texts = [message.text for message in after["messages"]]
print(texts)
assert all("learner@example.com" not in text for text in texts)

['회신: learner@****.com 입니다.', '확인: learner@****.com']


## 적용 범위 비교

고객 상담에서는 사용자가 보낸 주민등록번호가 모델로 들어가기 전 마스킹되어야 하고, 모델이 도구 결과에서 읽은 이메일도 최종 응답에서 다시 검사해야 합니다. 로그 저장 경로가 미들웨어 바깥에 있다면 로그에는 원문이 먼저 남을 수 있으므로 데이터가 지나가는 모든 경계를 따로 검토합니다.

마스킹은 값을 숨기면서 요청을 계속 처리하고, 차단은 요청을 중단합니다. 계좌 이체처럼 민감값 없이 업무를 수행할 수 없는 경우에는 무조건 마스킹보다 승인된 별도 경로가 필요할 수 있습니다. 이번 결과는 네 가지 예시 문자열과 지정한 감지 범위에서만 확인한 것입니다.


In [6]:
# 이 셀은 적용 범위 비교 작업을 수행하고 확인할 값을 출력합니다.
print("Ch16 직접 치환:", direct_masked)
print("Ch16 종류별 개수:", direct_counts)
print("미들웨어 입력:", after["messages"][0].text)
print("미들웨어 출력:", after["messages"][-1].text)
print("차이: 내장 email 한 유형만 부분 마스킹하며 Ch16의 네 유형·치환 문자열·개수 반환은 자동 보존하지 않습니다.")
print("estimated_cost_usd: 0.0 (결정적 로컬 모델)")

Ch16 직접 치환: <EMAIL_MASKED> <PHONE_MASKED> <RRN_MASKED> <CARD_MASKED>
Ch16 종류별 개수: {'email': 1, 'phone': 1, 'rrn': 1, 'card': 1}
미들웨어 입력: 회신: learner@****.com 입니다.
미들웨어 출력: 확인: learner@****.com
차이: 내장 email 한 유형만 부분 마스킹하며 Ch16의 네 유형·치환 문자열·개수 반환은 자동 보존하지 않습니다.
estimated_cost_usd: 0.0 (결정적 로컬 모델)
